# URBAN PULSE: 01 - NYC 311 API Data Extraction Pipeline
### Domain: Urban Services & Civic Operations Analytics
This notebook details the retrieval procedure for NYC 311 service request records via the NYC Open Data Socrata (SODA) API.

**Objectives:**
- Query endpoint `https://data.cityofnewyork.us/resource/erm2-nwe9.json`
- Select authoritative ~25 analytical schema fields
- Implement pagination using `$limit` and `$offset`
- Handle rate limits (HTTP 429) and network timeouts with exponential backoff
- Preserve raw snapshot in `data/raw_data.csv` and log metadata to `data/extraction_metadata.json`


In [1]:
import os
import sys
import time
import json
import requests
import pandas as pd

# Add parent directory to module search path
sys.path.append("..")
from src.extraction import NYC_311_ENDPOINT, SELECTED_FIELDS, build_headers, fetch_batch_with_retry


### 1. Selected Schema Fields
We specify 25 core analytical fields to optimize memory and network footprint.


In [2]:
print(f"Target API Endpoint: {NYC_311_ENDPOINT}")
print(f"Authoritative Selected Fields ({len(SELECTED_FIELDS)}):")
for i, f in enumerate(SELECTED_FIELDS, 1):
    print(f"  {i:02d}. {f}")


Target API Endpoint: https://data.cityofnewyork.us/resource/erm2-nwe9.json
Authoritative Selected Fields (25):
  01. unique_key
  02. created_date
  03. closed_date
  04. due_date
  05. resolution_action_updated_date
  06. agency
  07. agency_name
  08. complaint_type
  09. descriptor
  10. descriptor_2
  11. location_type
  12. incident_zip
  13. incident_address
  14. street_name
  15. city
  16. status
  17. resolution_description
  18. community_board
  19. council_district
  20. police_precinct
  21. borough
  22. open_data_channel_type
  23. park_borough
  24. latitude
  25. longitude


### 2. Live API Probe & Response Inspection
Demonstrate a single parameterized query to verify endpoint health and JSON structure.


In [3]:
headers = build_headers()
params = {
    "$select": ",".join(SELECTED_FIELDS),
    "$limit": 5,
    "$order": "created_date DESC"
}

try:
    response = requests.get(NYC_311_ENDPOINT, params=params, headers=headers, timeout=10)
    print("Status Code:", response.status_code)
    if response.status_code == 200:
        sample_records = response.json()
        print(f"Retrieved {len(sample_records)} sample records successfully.")
        display_df = pd.DataFrame(sample_records)[["unique_key", "created_date", "agency", "complaint_type", "borough"]]
        print(display_df)
    else:
        print("API Response:", response.text)
except Exception as e:
    print("API Connection note:", e)


API Connection note: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Max retries exceeded with url: /resource/erm2-nwe9.json?%24select=unique_key%2Ccreated_date%2Cclosed_date%2Cdue_date%2Cresolution_action_updated_date%2Cagency%2Cagency_name%2Ccomplaint_type%2Cdescriptor%2Cdescriptor_2%2Clocation_type%2Cincident_zip%2Cincident_address%2Cstreet_name%2Ccity%2Cstatus%2Cresolution_description%2Ccommunity_board%2Ccouncil_district%2Cpolice_precinct%2Cborough%2Copen_data_channel_type%2Cpark_borough%2Clatitude%2Clongitude&%24limit=5&%24order=created_date+DESC (Caused by NameResolutionError("HTTPSConnection(host='data.cityofnewyork.us', port=443): Failed to resolve 'data.cityofnewyork.us' ([Errno 11001] getaddrinfo failed)"))


### 3. Load & Audit the Preserved Raw Extract
Review the saved snapshot in `data/raw_data.csv`.


In [4]:
raw_data_path = "../data/raw_data.csv"
if os.path.exists(raw_data_path):
    df_raw = pd.read_csv(raw_data_path, low_memory=False)
    print(f"Raw Dataset Dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
    print("\nColumn Information:")
    print(df_raw.dtypes)
    print("\nFirst 3 Records:")
    print(df_raw[["unique_key", "created_date", "agency", "complaint_type", "borough"]].head(3))
else:
    print(f"File not found: {raw_data_path}")


Raw Dataset Dimensions: 81,115 rows x 25 columns

Column Information:
unique_key                          int64
created_date                          str
closed_date                           str
resolution_action_updated_date        str
agency                                str
agency_name                           str
complaint_type                        str
descriptor                            str
location_type                         str
incident_zip                      float64
incident_address                      str
street_name                           str
city                                  str
status                                str
resolution_description                str
community_board                       str
council_district                  float64
police_precinct                       str
borough                               str
open_data_channel_type                str
park_borough                          str
latitude                          float64
longit

### 4. Review Extraction Metadata
Audit extraction timestamp, parameters, realized row counts, and duration.


In [5]:
meta_path = "../data/extraction_metadata.json"
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=4))
else:
    print("No extraction metadata file found.")


{
    "extraction_date_utc": "2026-11-06T20:47:41.294903+00:00",
    "endpoint": "https://data.cityofnewyork.us/resource/erm2-nwe9.json",
    "target_records": 100000,
    "realized_records": 81115,
    "columns_count": 25,
    "selected_fields": [
        "unique_key",
        "created_date",
        "closed_date",
        "due_date",
        "resolution_action_updated_date",
        "agency",
        "agency_name",
        "complaint_type",
        "descriptor",
        "descriptor_2",
        "location_type",
        "incident_zip",
        "incident_address",
        "street_name",
        "city",
        "status",
        "resolution_description",
        "community_board",
        "council_district",
        "police_precinct",
        "borough",
        "open_data_channel_type",
        "park_borough",
        "latitude",
        "longitude"
    ],
    "batch_size": 5000,
    "batches_completed": 18,
    "duration_seconds": 464.67,
    "temporal_window_start": "2020-07-20 13:03:0